In [4]:
from bson import decode_file_iter
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

In [5]:
HEADER_FILE = "bson_file/header.bson"
ACC_FILE = "bson_file/accelerogram.bson"

In [6]:
print("=" * 60)
print("Reading header.bson")
print("=" * 60)

headers = {}

count = 0

with open(HEADER_FILE, "rb") as f:

    for doc in decode_file_iter(f):

        count += 1

        headers[doc["_id"]] = {
            "Station": doc["StationCode"],
            "SamplingRate": doc["SamplingRate_Hz"],
            "PIndex": int(doc["pIndex"]),
            "PGA": doc["PGA_gal"]
        }

print("Headers Loaded :", count)

Reading header.bson
Headers Loaded : 38030


In [7]:
print("=" * 60)
print("Reading accelerogram.bson")
print("=" * 60)

records = []

count = 0

with open(ACC_FILE, "rb") as f:

    for doc in decode_file_iter(f):

        records.append(doc)

        count += 1

print("Signal Records :", count)

Reading accelerogram.bson
Signal Records : 114090


In [8]:
print("=" * 60)
print("Grouping Signals")
print("=" * 60)

from collections import defaultdict

events = defaultdict(dict)

for doc in records:

    full_id = doc["_id"]

    event_id, component = full_id.rsplit(".", 1)

    events[event_id][component] = np.asarray(
        doc["accelerogram"],
        dtype=np.float32
    )

print("Total Events :", len(events))

Grouping Signals
Total Events : 38030


In [10]:
def normalize(signal):
    signal = np.asarray(signal, dtype=np.float32)

    mean = signal.mean()
    std = signal.std()

    if std == 0:
        std = 1.0

    return (signal - mean) / std


In [12]:
import pandas as pd

results = []

WINDOW_SIZE = 200
STEP_SIZE = 200

for event_id, comp in events.items():

    # Skip incomplete events
    if event_id not in headers:
        continue

    if not all(c in comp for c in ["UD", "NS", "EW"]):
        continue

    # ----------------------------
    # Trim channels
    # ----------------------------
    min_length = min(
        len(comp["UD"]),
        len(comp["NS"]),
        len(comp["EW"])
    )

    ud = comp["UD"][:min_length]
    ns = comp["NS"][:min_length]
    ew = comp["EW"][:min_length]

    # ----------------------------
    # Normalize
    # ----------------------------
    ud = normalize(ud)
    ns = normalize(ns)
    ew = normalize(ew)

    # ----------------------------
    # Create windows
    # ----------------------------
    windows = []

    for start in range(0, min_length - WINDOW_SIZE + 1, STEP_SIZE):

        end = start + WINDOW_SIZE

        window = np.stack([
            ud[start:end],
            ns[start:end],
            ew[start:end]
        ])

        windows.append(window)

    if len(windows) == 0:
        continue

    # ----------------------------
    # Convert to tensor
    # ----------------------------
    windows = np.array(windows)

    windows = np.transpose(windows, (0, 2, 1))

    windows_tensor = torch.from_numpy(windows).float()

    # ----------------------------
    # Prediction
    # ----------------------------
    with torch.no_grad():

        output = model(windows_tensor)

    probabilities = torch.exp(output)

    predictions = torch.argmax(probabilities, dim=1)

    # ----------------------------
    # Save result
    # ----------------------------
    results.append({
        "Event ID": event_id,
        "Window Predictions": predictions.tolist()
    })

# ----------------------------
# Final DataFrame
# ----------------------------
df = pd.DataFrame(results)

print(df)

NameError: name 'model' is not defined